In [1]:
from PIL import Image
import os
import json
os.environ["DISPLAY"] = ":0"

import torch
from torch import nn
from torchvision import models, transforms
from executorch.runtime import Runtime

W0701 01:19:02.104000 1654 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
runtime = Runtime.get()

In [3]:
program = runtime.load_program("model.pte")
model = program.load_method("forward")

[cpuinfo_utils.cpp:71] Reading file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:87] Failed to open midr file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:100] Reading file /sys/devices/system/cpu/cpu0/regs/identification/midr_el1
[cpuinfo_utils.cpp:100] Reading file /sys/devices/system/cpu/cpu1/regs/identification/midr_el1
[cpuinfo_utils.cpp:100] Reading file /sys/devices/system/cpu/cpu2/regs/identification/midr_el1
[cpuinfo_utils.cpp:100] Reading file /sys/devices/system/cpu/cpu3/regs/identification/midr_el1


In [4]:
!big-display off

In [5]:
classes = []
with open("classes.json", "r") as f:
    classes = json.load(f)

In [6]:
mobilenet_mean = [0.485, 0.456, 0.406]
mobilenet_std = [0.229, 0.224, 0.225]

validation_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mobilenet_mean,
        std=mobilenet_std,
    )
])

unnormalize_transform = transforms.Normalize(
    mean=[-m / s for m, s in zip(mobilenet_mean, mobilenet_std)],
    std=[1 / s for s in mobilenet_std]
)

In [7]:
from picamera2 import Picamera2, Preview, Platform

In [8]:
cam = Picamera2()

[0:01:34.883658430] [1654]  INFO Camera camera_manager.cpp:340 libcamera v0.7.1+rpt20260609
[0:01:34.915019723] [1722]  INFO RPI pipeline_base.cpp:1133 Using configuration file '/usr/share/libcamera/pipeline/rpi/vc4/rpi_apps.yaml'
[0:01:34.953202958] [1722]  INFO IPAProxy ipa_proxy.cpp:184 Using tuning file /usr/share/libcamera/ipa/rpi/vc4/imx219.json
[0:01:34.960613782] [1722]  INFO Camera camera_manager.cpp:223 Adding camera '/base/soc/i2c0mux/i2c@1/imx219@10' for pipeline handler rpi/vc4
[0:01:34.960662598] [1722]  INFO RPI vc4.cpp:445 Registered camera /base/soc/i2c0mux/i2c@1/imx219@10 to Unicam device /dev/media1 and ISP device /dev/media0


In [9]:
config = cam.create_still_configuration(main={
    "size": (224, 224),
    "format": "BGR888",
}, display="main")
cam.configure(config)
cam.set_controls({"FrameRate": 36})
cam.start_preview(Preview.QT)
cam.start()

[0:01:34.997770813] [1654]  INFO Camera camera.cpp:1216 configuring streams: (0) 224x224-BGR888/sRGB (1) 640x480-SBGGR10_CSI2P/RAW
[0:01:34.998225192] [1722]  INFO RPI vc4.cpp:620 Sensor: /base/soc/i2c0mux/i2c@1/imx219@10 - Selected sensor format: 640x480-SBGGR10_1X10/RAW - Selected unicam format: 640x480-pBAA/RAW
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-pi'


In [ ]:
# read frame
while True:
    img = cam.capture_image("main")
    input_tensor = validation_transforms(img)
    input_batch = input_tensor.unsqueeze(0)
    output = model.execute(input_batch)
    print(classes[torch.tensor(output[0]).argmax().item()], output[0])

In [ ]:
cam.stop()